# §13.3.5 — 헤드 가지치기가 드러내는 중복성

> 딥러닝 교재 · 3부 13장 3절 5항 (🐍)
> 선행: §13.3.1(차원 분할) · §13.3.2(저계수 제약) · §13.3.6(과잉 해석 경고)

## 이 노트북이 답하는 질문

1. **학습된 헤드들의 한계 기여는 균등한가?** 하나씩 제거하며 잰다.
2. **몇 개까지 지워도 버티는가?** 탐욕 가지치기 곡선으로 답한다.
3. **살아남는 헤드와 지워지는 헤드는 무엇이 다른가?** 패턴 상관과 엔트로피로 본다.

**예상 실행 시간** CPU 약 60초 (`FAST = True`이면 약 30초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제 — 헤드에게 일자리가 둘 이상 있는 분류

내용 토큰 16개의 열에서 한 위치는 A-표지판, 한 위치는 B-표지판이다(표지가 붙은
내용 토큰은 별도의 토큰 id를 갖는다). 마지막 질의 토큰 위치에서 정답
$(c_A+c_B)\bmod 16$을 맞혀야 한다. A를 찾는 조회와 B를 찾는 조회, **최소 두 개의
서로 다른 검색**이 필요하므로 헤드들이 분업할 무대가 있다 — 그리고 헤드가 8개면
일자리보다 헤드가 많다.

모델은 공용 미니 트랜스포머(1층, $h=8$, $d=64$)다. 완전한 순전파·역전파가 아래 셀에
포함되어 있다.

In [ ]:
# ── 공용 미니 트랜스포머 (NumPy, 완전한 순전파+역전파) ──────────
# 구조: 임베딩 → L × [Pre-LN 블록 (MHA + MLP, 잔차)] → LN → 판독
# 위치 부호화: 'learned' | 'sin' | 'rope' | 'alibi' | 'none'

def make_config(V, d=48, L=2, h=4, T_max=64, pe='learned', causal=True, seed=0):
    dh = d // h
    rn = np.random.default_rng(seed)
    p = {}
    p['emb'] = rn.standard_normal((V, d)) * 0.5 / np.sqrt(d)
    if pe == 'learned':
        p['pos'] = rn.standard_normal((T_max, d)) * 0.5 / np.sqrt(d)
    for l in range(L):
        s = f'l{l}_'
        for nm in ['wq', 'wk', 'wv', 'wo']:
            p[s + nm] = rn.standard_normal((d, d)) / np.sqrt(d)
        p[s + 'ln1g'] = np.ones(d); p[s + 'ln1b'] = np.zeros(d)
        p[s + 'w1'] = rn.standard_normal((d, 4 * d)) / np.sqrt(d)
        p[s + 'b1'] = np.zeros(4 * d)
        p[s + 'w2'] = rn.standard_normal((4 * d, d)) / np.sqrt(4 * d)
        p[s + 'b2'] = np.zeros(d)
        p[s + 'ln2g'] = np.ones(d); p[s + 'ln2b'] = np.zeros(d)
    p['lnfg'] = np.ones(d); p['lnfb'] = np.zeros(d)
    p['out'] = rn.standard_normal((d, V)) / np.sqrt(d)
    cfg = dict(V=V, d=d, L=L, h=h, dh=dh, T_max=T_max, pe=pe, causal=causal)
    return p, cfg

def _sin_pe(T, d):
    pos = np.arange(T)[:, None]
    l2 = np.arange(0, d, 2)[None, :]
    ang = pos / (10000.0 ** (l2 / d))
    pe = np.zeros((T, d))
    pe[:, 0::2] = np.sin(ang); pe[:, 1::2] = np.cos(ang)
    return pe

def _rope_angles(T, dh, scale=1.0):
    pos = np.arange(T)[:, None] * scale
    l2 = np.arange(0, dh, 2)[None, :]
    return pos / (10000.0 ** (l2 / dh))          # (T, dh/2)

def _rope_apply(x, ang, inverse=False):
    # x: (B,h,T,dh) — 짝수/홀수 쌍을 각도 ang(T,dh/2)만큼 회전
    c, s = np.cos(ang), np.sin(ang)
    if inverse:
        s = -s
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return np.stack([x1 * c - x2 * s, x1 * s + x2 * c], axis=-1).reshape(x.shape)

def _alibi_slopes(h):
    return np.array([2.0 ** (-8.0 * (i + 1) / h) for i in range(h)])

def _ln_f(x, g, b):
    mu = x.mean(-1, keepdims=True)
    xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    inv = 1.0 / np.sqrt(var + 1e-5)
    xh = xc * inv
    return xh * g + b, (xh, inv)

def _ln_b(dy, cache, g):
    xh, inv = cache
    dxh = dy * g
    dg = (dy * xh).sum(axis=tuple(range(dy.ndim - 1)))
    db = dy.sum(axis=tuple(range(dy.ndim - 1)))
    dx = inv * (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True))
    return dx, dg, db

def forward(p, cfg, idx, targets=None, rope_scale=1.0, head_mask=None,
            skip=None, want_attn=False, kv_keep=None):
    """idx:(B,T) 정수. targets:(B,T) 또는 None.
    head_mask:(L,h) 0/1, skip: {'attn':set(l), 'mlp':set(l)},
    kv_keep:(T,) bool — 열 s의 키·값 사용 여부(캐시 축출 흉내)."""
    B, T = idx.shape
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    x = p['emb'][idx]                              # (B,T,d)
    if cfg['pe'] == 'learned':
        x = x + p['pos'][:T]
    elif cfg['pe'] == 'sin':
        x = x + _sin_pe(T, d)
    cache = {'idx': idx, 'T': T, 'B': B, 'xs': [], 'attn': []}
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    if cfg['causal']:
        nmask = np.triu(np.full((T, T), -np.inf), k=1)
    else:
        nmask = np.zeros((T, T))
    if kv_keep is not None:
        nmask = nmask.copy()
        nmask[:, ~kv_keep] = -np.inf
    if cfg['pe'] == 'alibi':
        sl = _alibi_slopes(h)
        dist = np.maximum(np.arange(T)[:, None] - np.arange(T)[None, :], 0)
        abias = -sl[:, None, None] * dist[None]    # (h,T,T)
    else:
        abias = np.zeros((1, T, T))
    skip = skip or {'attn': set(), 'mlp': set()}
    attns = []
    def split(z):
        return z.reshape(B, T, h, dh).transpose(0, 2, 1, 3)       # (B,h,T,dh)
    for l in range(L):
        s = f'l{l}_'
        c = {}
        if l not in skip['attn']:
            # ── MHA 가지 ──
            h1, c['ln1'] = _ln_f(x, p[s + 'ln1g'], p[s + 'ln1b'])
            c['h1'] = h1
            q = split(h1 @ p[s + 'wq']); k = split(h1 @ p[s + 'wk']); v = split(h1 @ p[s + 'wv'])
            if cfg['pe'] == 'rope':
                q = _rope_apply(q, ang); k = _rope_apply(k, ang)
            e = np.einsum('bhtd,bhsd->bhts', q, k) / np.sqrt(dh) + nmask + abias[None]
            e -= e.max(-1, keepdims=True)
            a = np.exp(e); a /= a.sum(-1, keepdims=True)
            if head_mask is not None:
                hm = head_mask[l][None, :, None, None]
            else:
                hm = 1.0
            av = np.einsum('bhts,bhsd->bhtd', a, v) * hm
            avm = av.transpose(0, 2, 1, 3).reshape(B, T, d)
            x = x + avm @ p[s + 'wo']
            c.update(q=q, k=k, v=v, a=a, avm=avm, hm=hm)
            attns.append(a)
        else:
            attns.append(None)
        if l not in skip['mlp']:
            # ── MLP 가지 ──
            h2, c['ln2'] = _ln_f(x, p[s + 'ln2g'], p[s + 'ln2b'])
            z1 = h2 @ p[s + 'w1'] + p[s + 'b1']
            r = np.maximum(z1, 0.0)               # ReLU (역전파 단순화)
            x = x + r @ p[s + 'w2'] + p[s + 'b2']
            c['mlp'] = (h2, z1, r)
        else:
            c['mlp'] = None
        cache[s] = c
    hf, cache['lnf'] = _ln_f(x, p['lnfg'], p['lnfb'])
    cache['hf'] = hf
    logits = hf @ p['out']
    cache['logits'] = logits
    out = {'logits': logits}
    if want_attn:
        out['attn'] = attns
    if targets is not None:
        valid = targets >= 0                      # -1 = 손실에서 제외
        tsafe = np.maximum(targets, 0)
        z = logits - logits.max(-1, keepdims=True)
        lse = np.log(np.exp(z).sum(-1))
        ll = z[np.arange(B)[:, None], np.arange(T)[None, :], tsafe] - lse
        out['loss'] = -(ll * valid).sum() / max(valid.sum(), 1)
        P = np.exp(z); P /= P.sum(-1, keepdims=True)
        cache['P'] = P; cache['targets'] = tsafe; cache['valid'] = valid
    out['cache'] = cache
    return out

def backward(p, cfg, cache, rope_scale=1.0):
    B, T = cache['B'], cache['T']
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    g = {k: np.zeros_like(v) for k, v in p.items()}
    P, targets, valid = cache['P'], cache['targets'], cache['valid']
    dlogits = P.copy()
    dlogits[np.arange(B)[:, None], np.arange(T)[None, :], targets] -= 1.0
    dlogits *= valid[:, :, None]
    dlogits /= max(valid.sum(), 1)
    hf = cache['hf']
    g['out'] = np.einsum('btd,btv->dv', hf, dlogits)
    dhf = dlogits @ p['out'].T
    dx, g['lnfg'], g['lnfb'] = _ln_b(dhf, cache['lnf'], p['lnfg'])
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    for l in range(L - 1, -1, -1):
        s = f'l{l}_'
        c = cache[s]
        if c['mlp'] is not None:
            h2, z1, r = c['mlp']
            dmlp = dx                                   # 잔차: 가지로 흘러드는 기울기
            g[s + 'w2'] += np.einsum('btf,btd->fd', r, dmlp)
            g[s + 'b2'] += dmlp.sum((0, 1))
            dr = dmlp @ p[s + 'w2'].T
            dz1 = dr * (z1 > 0)
            g[s + 'w1'] += np.einsum('btd,btf->df', h2, dz1)
            g[s + 'b1'] += dz1.sum((0, 1))
            dh2 = dz1 @ p[s + 'w1'].T
            dxi, dg2, db2 = _ln_b(dh2, c['ln2'], p[s + 'ln2g'])
            g[s + 'ln2g'] += dg2; g[s + 'ln2b'] += db2
            dx = dx + dxi
        if 'a' not in c:
            continue
        # MHA 가지
        dattn_out = dx
        g[s + 'wo'] += np.einsum('btd,bte->de', c['avm'], dattn_out)
        davm = dattn_out @ p[s + 'wo'].T
        dav = davm.reshape(B, T, h, dh).transpose(0, 2, 1, 3) * c['hm']
        a, q, k, v = c['a'], c['q'], c['k'], c['v']
        da = np.einsum('bhtd,bhsd->bhts', dav, v)
        dv = np.einsum('bhts,bhtd->bhsd', a, dav)
        de = a * (da - (a * da).sum(-1, keepdims=True))
        dq = np.einsum('bhts,bhsd->bhtd', de, k) / np.sqrt(dh)
        dk = np.einsum('bhts,bhtd->bhsd', de, q) / np.sqrt(dh)
        if cfg['pe'] == 'rope':
            dq = _rope_apply(dq, ang, inverse=True)
            dk = _rope_apply(dk, ang, inverse=True)
        def merge(z):
            return z.transpose(0, 2, 1, 3).reshape(B, T, d)
        dq, dk, dv = merge(dq), merge(dk), merge(dv)
        h1 = c['h1']
        g[s + 'wq'] += np.einsum('btd,bte->de', h1, dq)
        g[s + 'wk'] += np.einsum('btd,bte->de', h1, dk)
        g[s + 'wv'] += np.einsum('btd,bte->de', h1, dv)
        dh1 = dq @ p[s + 'wq'].T + dk @ p[s + 'wk'].T + dv @ p[s + 'wv'].T
        dxi, dg1, db1 = _ln_b(dh1, c['ln1'], p[s + 'ln1g'])
        g[s + 'ln1g'] += dg1; g[s + 'ln1b'] += db1
        dx = dx + dxi
    if cfg['pe'] == 'learned':
        g['pos'][:T] += dx.sum(0)
    np.add.at(g['emb'], cache['idx'], dx)
    return g

def adam_init(p):
    return {k: np.zeros_like(v) for k, v in p.items()}, {k: np.zeros_like(v) for k, v in p.items()}

def adam_step(p, g, m, v, t, lr=3e-3):
    for k in p:
        m[k] = 0.9 * m[k] + 0.1 * g[k]
        v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2
        p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)

def train_lm(p, cfg, sample_batch, steps, lr=3e-3, log_every=0, rope_scale=1.0):
    m, v = adam_init(p)
    hist = []
    for t in range(1, steps + 1):
        idx, tgt = sample_batch()
        out = forward(p, cfg, idx, targets=tgt, rope_scale=rope_scale)
        g = backward(p, cfg, out['cache'], rope_scale=rope_scale)
        adam_step(p, g, m, v, t, lr)
        hist.append(out['loss'])
        if log_every and t % log_every == 0:
            print(f"  step {t}: loss {np.mean(hist[-log_every:]):.3f}")
    return hist

In [ ]:
NC = 16
V = 3 * NC + 1            # 내용 16 + A판 16 + B판 16 + 질의 토큰
QTOK = 3 * NC
T_SEQ = 17
N_HEADS = 8

def sample_batch(B, rn):
    cls = np.array([rn.permutation(NC) for _ in range(B)])
    pa = np.zeros(B, int); pb = np.zeros(B, int)
    for b in range(B):
        pa[b], pb[b] = rn.permutation(NC)[:2]
    idx = cls.copy()
    idx[np.arange(B), pa] = NC + cls[np.arange(B), pa]        # A-표지판
    idx[np.arange(B), pb] = 2 * NC + cls[np.arange(B), pb]    # B-표지판
    idx = np.concatenate([idx, np.full((B, 1), QTOK)], axis=1)
    y = (cls[np.arange(B), pa] + cls[np.arange(B), pb]) % NC
    tgt = np.full((B, T_SEQ), -1)
    tgt[:, -1] = y                                            # 손실은 질의 위치에서만
    return idx, tgt

p, cfg = make_config(V=V, d=64, L=1, h=N_HEADS, T_max=T_SEQ, pe='learned', seed=1)
rn = np.random.default_rng(SEED % 1000)
STEPS = 300 if FAST else 600
hist = train_lm(p, cfg, lambda: sample_batch(64, rn), steps=STEPS, lr=3e-3)
idx_ev, tgt_ev = sample_batch(512, np.random.default_rng(SEED + 9))
out = forward(p, cfg, idx_ev, targets=tgt_ev, want_attn=True)
acc0 = (out['cache']['P'][:, -1].argmax(1) == tgt_ev[:, -1]).mean()
base_loss = out['loss']
print(f"학습 후 정확도 {acc0:.3f} (우연 수준 {1/NC:.3f}) | 평가 손실 {base_loss:.3f}")

---
## 2. 한계 기여 — 하나씩 제거

헤드 $i$의 출력을 0으로 마스킹하고(`head_mask`) 손실 증가를 잰다.

In [ ]:
def eval_mask(hm):
    o = forward(p, cfg, idx_ev, targets=tgt_ev, head_mask=hm)
    acc = (o['cache']['P'][:, -1].argmax(1) == tgt_ev[:, -1]).mean()
    return o['loss'], acc

marginal = []
for hh in range(N_HEADS):
    hm = np.ones((1, N_HEADS)); hm[0, hh] = 0
    lo, ac = eval_mask(hm)
    marginal.append(lo - base_loss)
    print(f"헤드 {hh}: Δ손실 {lo-base_loss:+.3f}  정확도 {ac:.3f}")
marginal = np.array(marginal)

---
## 3. 탐욕 가지치기 — 몇 개까지 버티는가

남은 헤드들 중 제거 시 손실 증가가 가장 작은 것을 반복해서 지운다. 매 걸음에서
**다시** 잰다 — 헤드 간 상호작용(한 헤드를 지우면 다른 헤드의 기여가 변한다) 때문에
처음의 한계 기여 순서와 다를 수 있다.

In [ ]:
alive = list(range(N_HEADS))
prune_curve = [(0, base_loss, acc0)]
hm = np.ones((1, N_HEADS))
for k in range(1, N_HEADS):
    cand = []
    for hh in alive:
        hm2 = hm.copy(); hm2[0, hh] = 0
        lo, ac = eval_mask(hm2)
        cand.append((lo, ac, hh))
    lo, ac, hh = min(cand)
    hm[0, hh] = 0; alive.remove(hh)
    prune_curve.append((k, lo, ac))
    print(f"{k}개 제거 (마지막: 헤드 {hh}): 손실 {lo:.3f}  정확도 {ac:.3f}")
prune_curve = np.array(prune_curve)

---
## 4. 헤드는 서로 얼마나 겹치는 일을 하는가

질의 위치의 어텐션 분포(길이 $T$ 벡터)를 헤드마다 뽑아, 평가 배치 전체에 걸쳐
헤드 간 상관을 잰다. 상관 높은 두 헤드는 부분공간이 달라도 **하는 일이 같다**(§13.3.3).

In [ ]:
A_last = out['attn'][0][:, :, -1, :]          # (B, h, T) — 질의 위치의 분포
flat = A_last.transpose(1, 0, 2).reshape(N_HEADS, -1)
C = np.corrcoef(flat)
ent_h = -(A_last * np.log(A_last + 1e-12)).sum(-1).mean(0)   # 헤드별 엔트로피
print("헤드 간 상관행렬 (반올림):"); print(np.round(C, 2))

---
## 5. 교재 그림 — fig_13_3_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 한계 기여
ax = axes[0]
order = np.argsort(marginal)[::-1]
ax.bar(range(N_HEADS), marginal[order], color=[CB[5] if m > 0.1 else CB[1] for m in marginal[order]])
ax.set_xticks(range(N_HEADS)); ax.set_xticklabels(order)
ax.set_xlabel(lab('헤드 (기여 내림차순)', 'head (sorted)'))
ax.set_ylabel(lab('제거 시 손실 증가', '$\\Delta$ loss'))
ax.set_title(lab('(a) 한계 기여는 극단적으로 불균등하다', '(a) marginal contribution'), fontsize=10)

# (b) 탐욕 가지치기 곡선
ax = axes[1]
ax.plot(prune_curve[:, 0], prune_curve[:, 2], 'o-', color=CB[5], ms=5)
ax.axhline(1 / NC, color='k', lw=0.7, ls=':')
ax.text(0.2, 1 / NC + 0.03, lab('우연 수준', 'chance'), fontsize=8)
ax.set_xlabel(lab('제거한 헤드 수', 'heads removed'))
ax.set_ylabel(lab('정확도', 'accuracy'))
ax.set_title(lab('(b) 여덟 중 셋은 거의 공짜로 지워진다', '(b) greedy pruning'), fontsize=10)

# (c) 헤드 간 패턴 상관
ax = axes[2]
imv = ax.imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(imv, ax=ax, fraction=0.046)
ax.set_xticks(range(N_HEADS)); ax.set_yticks(range(N_HEADS))
ax.grid(False)
ax.set_xlabel(lab('헤드', 'head')); ax.set_ylabel(lab('헤드', 'head'))
ax.set_title(lab('(c) 어텐션 패턴의 헤드 간 상관', '(c) pattern correlation'), fontsize=10)

# (d) 엔트로피 대 기여
ax = axes[3]
ax.scatter(ent_h, marginal, s=45, c=[CB[5]] * N_HEADS, zorder=3)
for hh in range(N_HEADS):
    ax.annotate(str(hh), (ent_h[hh], marginal[hh]), fontsize=8,
                xytext=(4, 4), textcoords='offset points')
ax.set_xlabel(lab('헤드의 어텐션 엔트로피 (질의 위치)', 'head entropy'))
ax.set_ylabel(lab('제거 시 손실 증가', '$\\Delta$ loss'))
ax.set_title(lab('(d) 뾰족하게 조회하는 헤드가 일하는 헤드', '(d) entropy vs importance'), fontsize=10)

save_book_fig(fig, 'fig_13_3_5')
plt.show()

> ### 읽는 법
>
> (a) 여덟 헤드의 한계 기여는 극단적으로 불균등하다. 소수가 성능을 지탱하고,
> 두셋은 지워도 사실상 티가 나지 않는다 (Δ손실 0.00–0.03).
> (b) 탐욕 곡선은 세 개를 지울 때까지 거의 평평하다가(정확도 1.00 → 0.93), 일하는
> 헤드가 잘리기 시작하면 절벽처럼 떨어진다 — 큰 모델에서의 관찰과 같은 모양이다
> (Michel et al. 2019, Voita et al. 2019).
> (c) 이유의 일부가 상관에 있다. 패턴이 겹치는 헤드 무리는 서로를 대체할 수 있다.
> (d) 일하는 헤드는 질의 위치에서 뾰족하게(낮은 엔트로피) 조회하는 경향이 있다 —
> 다만 이 상관을 "역할"의 증거로 승격시키기 전에 §13.3.6의 경고를 다시 읽을 것.
> 다중 헤드는 분업 조직도라기보다, 학습이 쓸 만한 검색 몇 개를 건지도록 **복권을
> 여러 장 사는 것**에 가깝다.

---
## 6. 자기 점검

1. (a)의 순서로 지우는 것과 (b)의 탐욕(매 걸음 재평가) 순서가 다른가? 다르다면 헤드 간 어떤 상호작용을 뜻하는가?
2. 씨앗을 바꿔 다시 학습하면 "일하는 헤드"의 번호가 유지되는가? §13.3.6의 재현성 조건 (iii)과 연결하라.
3. 지운 헤드들을 되살리고 일하는 헤드 하나만 지운 뒤 **짧게 재학습**하면 성능이 회복되는가? 남은 헤드들이 역할을 넘겨받는지 관찰하라.
4. $h=2$로 줄여 다시 학습하면 (a)의 분포는 어떻게 변하는가? 일자리 수(2)와 헤드 수의 관계로 설명하라.

## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `N_HEADS` | 1절 | 8 | 일자리(2) 대비 잉여의 정도 |
| 표지 수 | 1절 | 2 (A·B) | 셋으로 늘리면 필수 헤드도 는다 |
| `STEPS` | 1절 | 600 | 덜 수렴한 모델의 중복성 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")